`07_RAG1.ipynb`

## <RAG의 순서>
## 1. Store(RAG 데이터 저장)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "/oss/python/langchain/agents",
    "/oss/python/deepagents/rag",
    "/oss/python/langchain/tools",
    "/oss/python/langchain/models",
    "/oss/python/deepagents/retrieval",
    "/oss/python/langchain/knowledge-base",
    "/oss/python/langchain/middleware",
    "/oss/python/deepagents/overview",
    "/oss/python/deepagents/subagents",
    "/oss/python/deepagents/streaming",
    "/oss/python/deepagents/frontend/subagent-streaming",
    "/oss/python/deepagents/backends",
    "/oss/python/langgraph/overview",
    "/oss/python/langgraph/quickstart",
]

In [ ]:
# 1. Load (PDF, HTML, TEXT, MD, IMG, VIDEO, HWPX(xml), XLSX, PPTX, DOCX) -> 문서 종류에 따라 방법이 다름

import requests
from langchain_core.documents import Document # RAG에 사용할 문서 쪼가리를 의미하는 데이터 타입

# Node, Edge 이런거 아님. 단순 함수
def load_langchain_docs():
  docs = []
  for path in DOC_PATHS:
      url = f'{DOCS_BASE}{path}.md'
      res = requests.get(url, timeout=10) # 5초간 답이 없으면 넘어가라
      doc = Document(page_content=res.text, metadata={'source':f'{DOCS_BASE}{path}'})#단순 str말고 Document 타입으로 잘 감싸기 -> RAG에 사용하기 위해
      docs.append(doc)
  return docs

docs = load_langchain_docs()
print(f'{len(docs)}개의 문서를 불러왔습니다')

In [ ]:
# 2. Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)#1000개 단위로 짜르되 문맥확인 위해서 위아래 200글자 겹치게

splits = splitter.split_documents(docs)
print(f'{len(splits)}개의 조각으로 분리')

In [ ]:
# 3. Embed
from langchain_openai import OpenAIEmbeddings

# embedding 담당자
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# 4. Store
from langchain_core.vectorstores import InMemoryVectorStore

# 벡터스토어 세팅(임베딩)
vectorstore = InMemoryVectorStore(embedding=embeddings)

# 벡터스토어 저장 (문서조각)
vectorstore.add_documents(documents=splits)  # 리턴값은 저장된 문서 ID (쓸모없음)
print('저장완료')

## 2. Retrieve(검색)

In [ ]:
## 확인
from pprint import pprint

for doc in vectorstore.similarity_search('RAG하는법'):
  pprint(doc.page_content)

In [7]:
# 1. Tool 만들기
from langchain.tools import tool


@tool(parse_docstring=True) # 'parse_docstring = True' -> docstring을 google style(밑의 description 구조)로 잘 작성했다면, Agent에게 구조화해서 보내 더 잘이해하게 됨
def rag_search_document(query: str):
    """Search LangChain documentation.

    Args:
        query: Natural language search query.
    """
    retrieved_docs = vectorstore.similarity_search(query, k=4)  # 문서 4개 검색

    result = '\n---\n'.join(map(lambda doc: doc.page_content, retrieved_docs))
    return result

In [ ]:
# 2. Agent 만들어서 Tool 쥐어주기
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')

agent = create_agent(
  model='openai:gpt-4.1-mini',
  tools=[rag_search_document],
  system_prompt="주어진 RAG 데이터를 tool로 조회해서 답변을 도출해내라",
)
#모든 에이전트는 graph인데 graph는 state가 들어가서 state가 나가는 형태이고 에이전트는 프리빌트된 messagesState를 쓰니까 나오고 들어가는 값의 키가 무조건 ['messages']이다 
result1 = agent.invoke({'messages': {'role':'user', 'content':'rag 하는법 에대해서 알려줘'}})#agent의 invoke값에 이렇게 넣는 이유
result2 = llm.invoke('안녕하세요')
print(result1['messages'][-1].content)
print(result2['messages'])

{'messages': [HumanMessage(content='rag 하는법 에대해서 알려줘', additional_kwargs={}, response_metadata={}, id='e8bdead6-5285-4a0f-a7bd-af4657637ad5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 75, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_bb832114e0', 'id': 'chatcmpl-ELiDL4rzaK2beeIwBV5lmrHderILV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07f62-79cc-7a43-904e-b2125cd2232d-0', tool_calls=[{'name': 'rag_search_document', 'args': {'query': 'RAG 하는법'}, 'id': 'call_rG0g

In [ ]:
print(result1['messages'][-1].content)
print('-------------------------')
print(result2)
print('-----------------')


RAG(Retrieval-Augmented Generation)은 외부 지식 기반을 활용하여 답변을 생성하는 기술입니다. RAG를 구현하는 방법은 다양하며, 주요 아키텍처로는 Hybrid RAG와 Agentic RAG가 있습니다.

1. Hybrid RAG
- 쿼리 개선: 입력 질문을 수정하여 검색 품질을 높임 (예: 질문 재작성, 다양성 생성, 문맥 확장)
- 검색 검증: 검색된 문서가 적절한지 평가, 부족하면 쿼리 재검색
- 답변 검증: 생성된 답변의 정확성, 완전성, 출처 정합성 검사
이 과정은 여러 번 반복될 수 있습니다.

2. Agentic RAG
- LLM 기반 에이전트가 단계별로 추론하며 언제, 어떻게 정보를 검색할지 결정
- 에이전트가 외부 지식 검색 도구(문서 로더, 웹 API, DB 쿼리 등)에 접근 가능해야 함

자세한 튜토리얼과 평가 방법은 LangChain 문서의 RAG 관련 페이지를 참고하면 도움이 됩니다.
-------------------------
content='안녕하세요! 무엇을 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 9, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider

langchain_core.messages.ai.AIMessage